# 08 — Extracción de frecuencias POS (Stanza)

Equivalente al notebook `10_extraer_frecuencias_POS.ipynb` de Karen.

Extrae lemas de verbos, adjetivos y sustantivos del subcorpus de salud usando Stanza.

**Entrada:** `salud_tweets_final.parquet`, `corpus_cleaned.parquet`  
**Salida:** `verbos_salud_stanza.parquet`, `adjetivos_salud_stanza.parquet`, `sustantivos_salud_stanza.parquet`

In [ ]:
# ============================================================
# CELL 0 — CONFIG
# ============================================================
from pathlib import Path

DATA_PROCESSED = Path(r'C:\Users\afpue\OneDrive\Documentos\GitHub\icare\kMetodo\resultadosPropios')

print('[CONFIG] OK')
print(f'  DATA_PROCESSED : {DATA_PROCESSED.resolve()}')


In [ ]:
# ============================================================
# CELL 1 — IMPORTS Y CARGA
# ============================================================
import pandas as pd
import stanza
from collections import Counter
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Corpus completo para obtener Texto_limpio
corpus = pd.read_parquet(DATA_PROCESSED / 'corpus_cleaned.parquet')

# Subcorpus de salud
tweets_salud = pd.read_parquet(DATA_PROCESSED / 'salud_tweets_final.parquet')

# Merge para obtener texto
df_salud = tweets_salud.merge(
    corpus[['id_doc', 'Texto_limpio']],
    on='id_doc', how='left'
)

# Columnas de interés
df_salud = df_salud[['id_doc', 'autor', 'fecha',
                      'etiqueta_salud', 'categoria_detectada',
                      'subcat_max', 'Texto_limpio']].copy()

print(f'Tweets de salud: {len(df_salud):,}')
df_salud.head(2)


In [ ]:
# ============================================================
# CELL 2 — DESCARGAR MODELO STANZA (solo primera vez)
# ============================================================
# stanza.download('es')   # descomentar si es la primera ejecución
print('Stanza listo. Si es la primera vez, descomenta stanza.download("es") arriba.')


In [ ]:
# ============================================================
# CELL 3 — INICIALIZAR PIPELINE
# ============================================================
nlp_stanza = stanza.Pipeline(
    lang="es",
    processors="tokenize,pos,lemma",
    use_gpu=False   # cambiar a True si hay GPU disponible
)
print('Pipeline Stanza inicializado.')


In [ ]:
# ============================================================
# CELL 4 — FUNCIÓN DE EXTRACCIÓN POS
# (Replica exacta de Karen)
# ============================================================
def extraer_pos_frecuencias_stanza(textos):
    """
    Procesa una lista de textos con Stanza.
    Devuelve tres listas de dicts {lema: frecuencia}:
      verbos_frec, adjetivos_frec, sustantivos_frec
    """
    verbos_frec, adjetivos_frec, sustantivos_frec = [], [], []

    for texto in tqdm(textos, desc="POS Stanza"):
        try:
            if not isinstance(texto, str) or not texto.strip():
                verbos_frec.append({})
                adjetivos_frec.append({})
                sustantivos_frec.append({})
                continue

            doc = nlp_stanza(texto)
            verbos, adjetivos, sustantivos = [], [], []

            for sent in doc.sentences:
                for w in sent.words:
                    if not w.lemma or not w.lemma.isalpha():
                        continue
                    upos = w.upos
                    lema = w.lemma.lower()
                    if upos == "VERB":
                        verbos.append(lema)
                    elif upos == "ADJ":
                        adjetivos.append(lema)
                    elif upos == "NOUN":
                        sustantivos.append(lema)

            verbos_frec.append(dict(Counter(verbos)))
            adjetivos_frec.append(dict(Counter(adjetivos)))
            sustantivos_frec.append(dict(Counter(sustantivos)))

        except Exception as e:
            print(f"Error: {e}")
            verbos_frec.append({})
            adjetivos_frec.append({})
            sustantivos_frec.append({})

    return verbos_frec, adjetivos_frec, sustantivos_frec


In [ ]:
# ============================================================
# CELL 5 — EJECUTAR EXTRACCIÓN
# (Puede tardar ~30-60 min dependiendo del hardware)
# ============================================================
textos = df_salud['Texto_limpio'].tolist()
verbos_frec, adjetivos_frec, sustantivos_frec = extraer_pos_frecuencias_stanza(textos)

print(f'Extracción completada para {len(textos):,} tweets.')


In [ ]:
# ============================================================
# CELL 6 — CONSTRUIR DATAFRAMES Y GUARDAR
# ============================================================

# Columnas base que se repiten en los tres DataFrames
cols_base = ['id_doc', 'autor', 'fecha', 'etiqueta_salud',
             'categoria_detectada', 'subcat_max']

# Verbos
df_verbos = df_salud[cols_base].copy()
df_verbos['verbos_lemas_frecuencias'] = verbos_frec
df_verbos.to_parquet(DATA_PROCESSED / 'verbos_salud_stanza.parquet', index=False)
print('[GUARDADO] verbos_salud_stanza.parquet')

# Adjetivos
df_adjetivos = df_salud[cols_base].copy()
df_adjetivos['adjetivos_lemas_frecuencias'] = adjetivos_frec
df_adjetivos.to_parquet(DATA_PROCESSED / 'adjetivos_salud_stanza.parquet', index=False)
print('[GUARDADO] adjetivos_salud_stanza.parquet')

# Sustantivos
df_sustantivos = df_salud[cols_base].copy()
df_sustantivos['sustantivos_lemas_frecuencias'] = sustantivos_frec
df_sustantivos.to_parquet(DATA_PROCESSED / 'sustantivos_salud_stanza.parquet', index=False)
print('[GUARDADO] sustantivos_salud_stanza.parquet')


In [ ]:
# ============================================================
# CELL 7 — VERIFICACIÓN RÁPIDA
# ============================================================
# Top 20 verbos más frecuentes en el subcorpus de salud
from collections import Counter

total_verbos = Counter()
for d in verbos_frec:
    total_verbos.update(d)

total_sust = Counter()
for d in sustantivos_frec:
    total_sust.update(d)

total_adj = Counter()
for d in adjetivos_frec:
    total_adj.update(d)

print('Top 20 VERBOS:')
print(total_verbos.most_common(20))
print()
print('Top 20 SUSTANTIVOS:')
print(total_sust.most_common(20))
print()
print('Top 20 ADJETIVOS:')
print(total_adj.most_common(20))

print()
print('Notebook 08 completado.')
print('Siguiente -> 09_descriptivos_subcategorias_salud.ipynb')
